In [2]:
import json
import math
import random
from typing import List, Dict, Any, Tuple

def compute_size(n: int, p: float, rounding: str = "ceil") -> int:
    if p <= 0:
        return 0
    if rounding == "ceil":
        return min(n, math.ceil(p * n))
    elif rounding == "floor":
        return min(n, math.floor(p * n))
    elif rounding == "round":
        return min(n, round(p * n))
    else:
        raise ValueError("rounding must be 'ceil', 'floor' or 'round'")

def split_cumulative(data: List[Dict[str, Any]],
                     percentages: List[float] = [0.10, 0.25, 0.50, 1.00],
                     shuffle: bool = False,
                     seed: int | None = None,
                     rounding: str = "ceil") -> Dict[str, List[Dict[str, Any]]]:
    """
    Trả về dictionary mapping "10%" -> subset (prefix 10%), "25%" -> prefix 25%, ...
    Nếu shuffle=True thì xáo dữệt trước khi lấy prefix (sử dụng seed nếu có).
    """
    n = len(data)
    if shuffle:
        rnd = random.Random(seed)
        data_copy = data[:]  # sao chép
        rnd.shuffle(data_copy)
    else:
        data_copy = data

    results: Dict[str, List[Dict[str, Any]]] = {}
    for p in percentages:
        size = compute_size(n, p, rounding=rounding)
        key = f"{int(p*100)}%"
        results[key] = data_copy[:size]
    return results

def save_splits_to_files(splits: Dict[str, List[Dict[str, Any]]],
                         prefix: str = "dataset_",
                         indent: int = 2):
    for key, subset in splits.items():
        filename = f"{prefix}{key}.json".replace("%", "")
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(subset, f, ensure_ascii=False, indent=indent)
        print(f"Saved {len(subset)} items to {filename}")

# Ví dụ sử dụng:
if __name__ == "__main__":
    # 1) Nếu bạn có file JSON (một mảng các object) như courses.json:
    with open("courses-20250810_110716.json", "r", encoding="utf-8") as f:
        data = json.load(f)

    

    # Gọi hàm để chia:
    splits = split_cumulative(data,
                              percentages=[0.05, 0.10, 0.25, 0.50, 1.00],
                              shuffle=False,  # True nếu muốn xáo trộn
                              seed=42,        # chỉ dùng khi shuffle=True
                              rounding="ceil")  # 'ceil', 'floor' hoặc 'round'

    # In kích thước các split
    for k, v in splits.items():
        print(k, "->", len(v), "items")

    # Lưu ra file nếu muốn
    save_splits_to_files(splits, prefix="courses_")

5% -> 73 items
10% -> 145 items
25% -> 362 items
50% -> 723 items
100% -> 1445 items
Saved 73 items to courses_5.json
Saved 145 items to courses_10.json
Saved 362 items to courses_25.json
Saved 723 items to courses_50.json
Saved 1445 items to courses_100.json
